# Lezione 11 — Pratica: ETL e pipeline di dati

Nella lezione di teoria abbiamo costruito una mini pipeline ETL completa: Extract (dati grezzi simulati), Transform (pulizia con pandas) e Load (`to_sql` in una tabella sqlite). Ora aggiungerai una fase di trasformazione aggiuntiva prima del caricamento.

## Esercizio: una fase di trasformazione aggiuntiva

Partendo da `datos_limpios` già costruito (Extract + la pulizia di base sono già fatti per te), aggiungi una colonna derivata `importe_con_descuento` che applichi uno sconto del 10% (`importe * 0.9`) **solo** alle righe dove `producto` è `"Camiseta"`. Gli altri importi devono restare invariati. Poi, carica la tabella risultante in sqlite e verificala con una query `SELECT`.

**Suggerimenti:**
- Puoi usare l'indicizzazione booleana di pandas: `datos_limpios.loc[condicion, "columna"]`.
- Inizia copiando la colonna `importe` così com'è nella nuova colonna, e poi sovrascrivi solo le righe che soddisfano la condizione.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# Extract (già risolto)
datos_crudos = pd.DataFrame({
    "id_venta":  [1, 2, 3, 4, 4, 5],
    "cliente":   ["Ana Torres", "luis peña", "CARLA RUIZ", "Diego Soto", "Diego Soto", None],
    "producto":  ["Camiseta", "camiseta", "Auriculares", None, None, "Mochila"],
    "importe":   [15.99, 15.99, 39.90, 25.00, 25.00, np.nan],
})

# Transform di base (già risolto)
datos_limpios = datos_crudos.drop_duplicates()
datos_limpios = datos_limpios.dropna(subset=["cliente"])
datos_limpios["importe"] = datos_limpios["importe"].fillna(0)
datos_limpios["cliente"] = datos_limpios["cliente"].str.title()
datos_limpios["producto"] = datos_limpios["producto"].fillna("Sin especificar").str.title()
datos_limpios["importe"] = pd.to_numeric(datos_limpios["importe"], errors="coerce").fillna(0)
datos_limpios = datos_limpios[datos_limpios["importe"] >= 0]

print("Dati puliti, prima della tua trasformazione aggiuntiva:")
print(datos_limpios)

In [ ]:
# TODO: crea la colonna "importe_con_descuento" applicando uno sconto
# del 10% solo alle righe dove producto == "Camiseta".
# Le altre righe devono conservare lo stesso importo originale.

# datos_limpios["importe_con_descuento"] = ...

# print(datos_limpios)

In [ ]:
# TODO: carica datos_limpios (già con la nuova colonna) in una tabella
# "ventas" di un database sqlite in memoria, e verifica il
# risultato con una query SELECT.

# conexion = sqlite3.connect(':memory:')
# datos_limpios.to_sql("ventas", conexion, index=False, if_exists="replace")
# print(pd.read_sql_query("SELECT * FROM ventas", conexion))
# conexion.close()

<details>
<summary>💡 Vedi soluzione</summary>

```python
# Nuova colonna: di default, lo stesso importo...
datos_limpios["importe_con_descuento"] = datos_limpios["importe"]

# ...e sovrascriviamo solo le righe di "Camiseta" con il 10% di sconto.
es_camiseta = datos_limpios["producto"] == "Camiseta"
datos_limpios.loc[es_camiseta, "importe_con_descuento"] = datos_limpios.loc[es_camiseta, "importe"] * 0.9

print(datos_limpios)

# Load con la nuova colonna già inclusa
conexion = sqlite3.connect(':memory:')
datos_limpios.to_sql("ventas", conexion, index=False, if_exists="replace")
print(pd.read_sql_query("SELECT * FROM ventas", conexion))
conexion.close()
```
</details>

## Per continuare

Torna al notebook di teoria se vuoi ripassare i tre passi dell'ETL prima di provare l'esercizio: [clase11_warehouses_etl_pipelines_IT.ipynb](../notebooks_clase/clase11_warehouses_etl_pipelines_IT.ipynb)